# GO UI UPDATED - WGo.js version

This is the edited `go2.ipynb`. Run all cells, then use the final `show_go_ui()` cell to launch the board.


In [1]:
import random, math, copy
from collections import deque

N = 9
EMPTY, BLACK, WHITE = 0, 1, -1
KOMI = 2.5

In [2]:
class GoGame:
    def __init__(self):
        self.board = [[EMPTY]*N for _ in range(N)]
        self.player = BLACK
        self.previous_board = None
        self.last_move = None
        self.captures = {BLACK: 0, WHITE: 0}
        self.game_over = False

    def copy(self):
        g = GoGame()
        g.board = [r[:] for r in self.board]
        g.player = self.player
        g.previous_board = None if self.previous_board is None else [r[:] for r in self.previous_board]
        g.last_move = self.last_move
        g.captures = self.captures.copy()
        g.game_over = self.game_over
        return g

    def neighbors(self, r, c):
        for dr, dc in [(1,0),(-1,0),(0,1),(0,-1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < N and 0 <= nc < N:
                yield nr, nc

    def group_and_liberties(self, r, c):
        color = self.board[r][c]
        group, libs = set(), set()
        q = deque([(r,c)])
        group.add((r,c))

        while q:
            x, y = q.popleft()
            for nx, ny in self.neighbors(x, y):
                if self.board[nx][ny] == EMPTY:
                    libs.add((nx, ny))
                elif self.board[nx][ny] == color and (nx, ny) not in group:
                    group.add((nx, ny))
                    q.append((nx, ny))

        return group, libs

    def boards_equal(self, a, b):
        return a is not None and b is not None and a == b

    def is_legal(self, r, c):
        if self.game_over:
            return False
        if not (0 <= r < N and 0 <= c < N):
            return False
        if self.board[r][c] != EMPTY:
            return False

        test = self.copy()
        old_board = [row[:] for row in test.board]
        test.board[r][c] = test.player
        opponent = -test.player
        captured = 0

        for nr, nc in test.neighbors(r, c):
            if test.board[nr][nc] == opponent:
                group, libs = test.group_and_liberties(nr, nc)
                if len(libs) == 0:
                    captured += len(group)
                    for x, y in group:
                        test.board[x][y] = EMPTY

        own_group, own_libs = test.group_and_liberties(r, c)
        if len(own_libs) == 0:
            return False

        if self.boards_equal(test.board, self.previous_board):
            return False

        return True

    def legal_moves(self):
        return [(r,c) for r in range(N) for c in range(N) if self.is_legal(r,c)]

    def play(self, r, c):
        if not self.is_legal(r, c):
            return False

        old_board = [row[:] for row in self.board]
        self.board[r][c] = self.player
        opponent = -self.player

        for nr, nc in self.neighbors(r, c):
            if self.board[nr][nc] == opponent:
                group, libs = self.group_and_liberties(nr, nc)
                if len(libs) == 0:
                    self.captures[self.player] += len(group)
                    for x, y in group:
                        self.board[x][y] = EMPTY

        self.previous_board = old_board
        self.last_move = (r, c)
        self.player *= -1
        return True

    def pass_turn(self):
        self.game_over = True

    def score(self):
        visited = set()
        black_score = sum(cell == BLACK for row in self.board for cell in row)
        white_score = sum(cell == WHITE for row in self.board for cell in row) + KOMI

        for r in range(N):
            for c in range(N):
                if self.board[r][c] != EMPTY or (r,c) in visited:
                    continue

                region, borders = set(), set()
                q = deque([(r,c)])
                visited.add((r,c))

                while q:
                    x,y = q.popleft()
                    region.add((x,y))
                    for nx, ny in self.neighbors(x,y):
                        val = self.board[nx][ny]
                        if val == EMPTY and (nx,ny) not in visited:
                            visited.add((nx,ny))
                            q.append((nx,ny))
                        elif val != EMPTY:
                            borders.add(val)

                if borders == {BLACK}:
                    black_score += len(region)
                elif borders == {WHITE}:
                    white_score += len(region)

        winner = BLACK if black_score > white_score else WHITE
        return black_score, white_score, winner

In [3]:
class SafeAgent:
    def __init__(self, policy):
        self.policy = policy

    def select_move(self, game):
        legal = game.legal_moves()
        if not legal:
            return None

        move = self.policy(game.copy())

        if move in legal:
            return move

        return random.choice(legal)

In [4]:
class MCTSBot:
    def __init__(self, simulations=150):
        self.simulations = simulations

    def __call__(self, game):
        legal = game.legal_moves()
        if not legal:
            return None

        scores = {m: 0 for m in legal}
        visits = {m: 1 for m in legal}
        root_player = game.player

        for _ in range(self.simulations):
            move = random.choice(legal)
            sim = game.copy()
            sim.play(*move)

            for _ in range(60):
                moves = sim.legal_moves()
                if not moves:
                    break
                sim.play(*random.choice(moves))

            b, w, winner = sim.score()
            visits[move] += 1
            if winner == root_player:
                scores[move] += 1

        return max(legal, key=lambda m: scores[m] / visits[m])

In [5]:
from IPython.display import HTML, display
import html as _html

GO_APP_HTML = '\n<!doctype html>\n<html>\n<head>\n  <meta charset="utf-8">\n  <script>\n/*! MIT license, more info: wgo.waltheri.net */(function(v,q){var m=document.getElementsByTagName("script"),g={version:"2.3.1",B:1,W:-1,ERROR_REPORT:!0,DIR:m[m.length-1].src.split("?")[0].split("/").slice(0,-1).join("/")+"/",lang:"en",i18n:{en:{}}};g.opera=-1!=navigator.userAgent.search(/(opera)(?:.*version)?[ \\/]([\\w.]+)/i);g.webkit=-1!=navigator.userAgent.search(/(webkit)[ \\/]([\\w.]+)/i);g.msie=-1!=navigator.userAgent.search(/(msie) ([\\w.]+)/i);g.mozilla=-1!=navigator.userAgent.search(/(mozilla)(?:.*? rv:([\\w.]+))?/i)&&!g.webkit&&!g.msie;g.t=\nfunction(a){var b=g.i18n[g.lang][a]||g.i18n.en[a];if(b){for(var c=1;c<arguments.length;c++)b=b.replace("$",arguments[c]);return b}return a};g.extendClass=function(a,b){b.prototype=Object.create(a.prototype);b.prototype.constructor=b;b.prototype.super=a;return b};g.abstractMethod=function(){throw Error("unimplemented abstract method");};g.clone=function(a){if(a&&"object"==typeof a){var b=a.constructor==Array?[]:{},c;for(c in a)b[c]=a[c]==a?a:g.clone(a[c]);return b}return a};g.filterHTML=function(a){return a&&\n"string"==typeof a?a.replace(/</g,"&lt;").replace(/>/g,"&gt;"):a};var h=function(a,b){b=b||{};for(var c in b)this[c]=b[c];for(c in g.Board.default)this[c]===q&&(this[c]=g.Board.default[c]);for(c in h.themes.default)this.theme[c]===q&&(this.theme[c]=h.themes.default[c]);this.tx=this.section.left;this.ty=this.section.top;this.bx=this.size-1-this.section.right;this.by=this.size-1-this.section.bottom;this.init();a.appendChild(this.element);this.pixelRatio=v.devicePixelRatio||1;this.width&&this.height?\nthis.setDimensions(this.width,this.height):this.width?this.setWidth(this.width):this.height&&this.setHeight(this.height)};h.themes={};h.themes.old={shadowColor:"rgba(32,32,32,0.5)",shadowTransparentColor:"rgba(32,32,32,0)",shadowBlur:0,shadowSize:function(a){return a.shadowSize},markupBlackColor:"rgba(255,255,255,0.8)",markupWhiteColor:"rgba(0,0,0,0.8)",markupNoneColor:"rgba(0,0,0,0.8)",markupLinesWidth:function(a){return a.autoLineWidth?a.stoneRadius/7:a.lineWidth},gridLinesWidth:1,gridLinesColor:function(a){return"rgba(0,0,0,"+\nMath.min(1,a.stoneRadius/15)+")"},starColor:"#000",starSize:function(a){return a.starSize*(a.width/300+1)},stoneSize:function(a){return a.stoneSize*Math.min(a.fieldWidth,a.fieldHeight)/2},coordinatesColor:"rgba(0,0,0,0.7)",font:function(a){return a.font},linesShift:.5};h.themes.default={shadowColor:"rgba(62,32,32,0.5)",shadowTransparentColor:"rgba(62,32,32,0)",shadowBlur:function(a){return.1*a.stoneRadius},shadowSize:1,markupBlackColor:"rgba(255,255,255,0.9)",markupWhiteColor:"rgba(0,0,0,0.7)",markupNoneColor:"rgba(0,0,0,0.7)",\nmarkupLinesWidth:function(a){return a.stoneRadius/8},gridLinesWidth:function(a){return a.stoneRadius/15},gridLinesColor:"#654525",starColor:"#531",starSize:function(a){return a.stoneRadius/8+1},stoneSize:function(a){return Math.min(a.fieldWidth,a.fieldHeight)/2},coordinatesColor:"#531",variationColor:"rgba(0,32,128,0.8)",font:"calibri",linesShift:.25};var k=function(a,b){return"function"==typeof b.theme[a]?b.theme[a](b):b.theme[a]},m={draw:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius;\nthis.beginPath();var f=k("shadowBlur",b),d=Math.max(0,d-.5),n=this.createRadialGradient(c-b.ls,e-b.ls,d-1-f,c-b.ls,e-b.ls,d+f);n.addColorStop(0,k("shadowColor",b));n.addColorStop(1,k("shadowTransparentColor",b));this.fillStyle=n;this.arc(c-b.ls,e-b.ls,d+f,0,2*Math.PI,!0);this.fill()},clear:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius;this.clearRect(c-1.1*d-b.ls,e-1.1*d-b.ls,2.2*d,2.2*d)}},p=function(a,b,c){return a.obj_arr[b][c][0].c==g.B?k("markupBlackColor",a):a.obj_arr[b][c][0].c==\ng.W?k("markupWhiteColor",a):k("markupNoneColor",a)},D=function(a,b,c){return a.obj_arr[b][c][0]&&a.obj_arr[b][c][0].c==g.W||a.obj_arr[b][c][0].c==g.B},w,y=function(a){for(var b=a.angle,c=a.angle,e=0;e<a.lines.length;e++){var b=b+a.lines[e],c=c-a.lines[e],d=a.ctx,f=a.x,n=a.y,g=a.radius,h=b,k=c,m=a.factor,l=a.thickness;d.strokeStyle="rgba(64,64,64,0.2)";d.lineWidth=g/30*l;d.beginPath();var g=g-Math.max(1,d.lineWidth),l=f+g*Math.cos(h*Math.PI),h=n+g*Math.sin(h*Math.PI),f=f+g*Math.cos(k*Math.PI),n=n+\ng*Math.sin(k*Math.PI),r=k=void 0,r=k=void 0;f>l?(k=(n-h)/(f-l),r=Math.atan(k)):f==l?r=Math.PI/2:(k=(n-h)/(f-l),r=Math.atan(k)-Math.PI);g*=m;k=Math.sin(r)*g;r=Math.cos(r)*g;g=l+k;m=h-r;k=f+k;r=n-r;d.moveTo(l,h);d.bezierCurveTo(g,m,k,r,f,n);d.stroke()}};h.drawHandlers={NORMAL:{stone:{draw:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius,f;a.c==g.W?(f=this.createRadialGradient(c-2*d/5,e-2*d/5,d/3,c-d/5,e-d/5,5*d/5),f.addColorStop(0,"#fff"),f.addColorStop(1,"#aaa")):(f=this.createRadialGradient(c-\n2*d/5,e-2*d/5,1,c-d/5,e-d/5,4*d/5),f.addColorStop(0,"#666"),f.addColorStop(1,"#000"));this.beginPath();this.fillStyle=f;this.arc(c-b.ls,e-b.ls,Math.max(0,d-.5),0,2*Math.PI,!0);this.fill()}},shadow:m},PAINTED:{stone:{draw:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius,f;a.c==g.W?(f=this.createRadialGradient(c-2*d/5,e-2*d/5,2,c-d/5,e-d/5,4*d/5),f.addColorStop(0,"#fff"),f.addColorStop(1,"#ddd")):(f=this.createRadialGradient(c-2*d/5,e-2*d/5,1,c-d/5,e-d/5,4*d/5),f.addColorStop(0,"#111"),\nf.addColorStop(1,"#000"));this.beginPath();this.fillStyle=f;this.arc(c-b.ls,e-b.ls,Math.max(0,d-.5),0,2*Math.PI,!0);this.fill();this.beginPath();this.lineWidth=d/6;a.c==g.W?(this.strokeStyle="#999",this.arc(c+d/8,e+d/8,d/2,0,Math.PI/2,!1)):(this.strokeStyle="#ccc",this.arc(c-d/8,e-d/8,d/2,Math.PI,1.5*Math.PI));this.stroke()}},shadow:m},GLOW:{stone:{draw:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius,f;a.c==g.W?(f=this.createRadialGradient(c-2*d/5,e-2*d/5,d/3,c-d/5,e-d/5,8*d/5),f.addColorStop(0,\n"#fff"),f.addColorStop(1,"#666")):(f=this.createRadialGradient(c-2*d/5,e-2*d/5,1,c-d/5,e-d/5,3*d/5),f.addColorStop(0,"#555"),f.addColorStop(1,"#000"));this.beginPath();this.fillStyle=f;this.arc(c-b.ls,e-b.ls,Math.max(0,d-.5),0,2*Math.PI,!0);this.fill()}},shadow:m},SHELL:{stone:{draw:function(a,b){var c,e,d=b.stoneRadius;w=w||Math.ceil(9999999*Math.random());c=b.getX(a.x);e=b.getY(a.y);var f;f=a.c==g.W?"#aaa":"#000";this.beginPath();this.fillStyle=f;this.arc(c-b.ls,e-b.ls,Math.max(0,d-.5),0,2*Math.PI,\n!0);this.fill();if(a.c==g.W){f=w%(3+a.x*b.size+a.y)%3;var n=b.size*b.size+a.x*b.size+a.y,n=2/n*(w%n);0==f?y({ctx:this,x:c,y:e,radius:d,angle:n,lines:[.1,.12,.11,.1,.09,.09,.09,.09],factor:.25,thickness:1.75}):1==f?y({ctx:this,x:c,y:e,radius:d,angle:n,lines:[.1,.09,.08,.07,.06,.06,.06,.06,.06,.06,.06],factor:.2,thickness:1.5}):y({ctx:this,x:c,y:e,radius:d,angle:n,lines:[.12,.14,.13,.12,.12,.12],factor:.3,thickness:2});f=this.createRadialGradient(c-2*d/5,e-2*d/5,d/3,c-d/5,e-d/5,5*d/5);f.addColorStop(0,\n"rgba(255,255,255,0.9)");f.addColorStop(1,"rgba(255,255,255,0)")}else f=this.createRadialGradient(c+.4*d,e+.4*d,0,c+.5*d,e+.5*d,d),f.addColorStop(0,"rgba(32,32,32,1)"),f.addColorStop(1,"rgba(0,0,0,0)"),this.beginPath(),this.fillStyle=f,this.arc(c-b.ls,e-b.ls,Math.max(0,d-.5),0,2*Math.PI,!0),this.fill(),f=this.createRadialGradient(c-.4*d,e-.4*d,1,c-.5*d,e-.5*d,1.5*d),f.addColorStop(0,"rgba(64,64,64,1)"),f.addColorStop(1,"rgba(0,0,0,0)");this.beginPath();this.fillStyle=f;this.arc(c-b.ls,e-b.ls,Math.max(0,\nd-.5),0,2*Math.PI,!0);this.fill()}},shadow:m},MONO:{stone:{draw:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius,f=k("markupLinesWidth",b)||1;this.fillStyle=a.c==g.W?"white":"black";this.beginPath();this.arc(c,e,Math.max(0,d-f),0,2*Math.PI,!0);this.fill();this.lineWidth=f;this.strokeStyle="black";this.stroke()}}},CR:{stone:{draw:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius;this.strokeStyle=a.c||p(b,a.x,a.y);this.lineWidth=a.lineWidth||k("markupLinesWidth",b)||1;this.beginPath();\nthis.arc(c-b.ls,e-b.ls,d/2,0,2*Math.PI,!0);this.stroke()}}},LB:{stone:{draw:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius,f=a.font||k("font",b)||"";this.fillStyle=a.c||p(b,a.x,a.y);this.font=1==a.text.length?Math.round(1.5*d)+"px "+f:2==a.text.length?Math.round(1.2*d)+"px "+f:Math.round(d)+"px "+f;this.beginPath();this.textBaseline="middle";this.textAlign="center";this.fillText(a.text,c,e,2*d)}},grid:{draw:function(a,b){if(!D(b,a.x,a.y)&&!a._nodraw){var c=b.getX(a.x),e=b.getY(a.y),\nd=b.stoneRadius;this.clearRect(c-d,e-d,2*d,2*d)}},clear:function(a,b){if(!D(b,a.x,a.y)){a._nodraw=!0;var c;b.grid.clear();b.grid.draw(b);for(var e=0;e<b.size;e++)for(var d=0;d<b.size;d++)for(var f=0;f<b.obj_arr[e][d].length;f++){var g=b.obj_arr[e][d][f];c=g.type?"string"==typeof g.type?h.drawHandlers[g.type]:g.type:b.stoneHandler;c.grid&&c.grid.draw.call(b.grid.getContext(g),g,b)}for(e=0;e<b.obj_list.length;e++)g=b.obj_list[e],c=g.handler,c.grid&&c.grid.draw.call(b.grid.getContext(g.args),g.args,\nb);delete a._nodraw}}}},SQ:{stone:{draw:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=Math.round(b.stoneRadius);this.strokeStyle=a.c||p(b,a.x,a.y);this.lineWidth=a.lineWidth||k("markupLinesWidth",b)||1;this.beginPath();this.rect(Math.round(c-d/2)-b.ls,Math.round(e-d/2)-b.ls,d,d);this.stroke()}}},TR:{stone:{draw:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius;this.strokeStyle=a.c||p(b,a.x,a.y);this.lineWidth=a.lineWidth||k("markupLinesWidth",b)||1;this.beginPath();this.moveTo(c-b.ls,\ne-b.ls-Math.round(d/2));this.lineTo(Math.round(c-d/2)-b.ls,Math.round(e+d/3)+b.ls);this.lineTo(Math.round(c+d/2)+b.ls,Math.round(e+d/3)+b.ls);this.closePath();this.stroke()}}},MA:{stone:{draw:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius;this.strokeStyle=a.c||p(b,a.x,a.y);this.lineCap="round";this.lineWidth=2*(a.lineWidth||k("markupLinesWidth",b)||1)-1;this.beginPath();this.moveTo(Math.round(c-d/2),Math.round(e-d/2));this.lineTo(Math.round(c+d/2),Math.round(e+d/2));this.moveTo(Math.round(c+\nd/2)-1,Math.round(e-d/2));this.lineTo(Math.round(c-d/2)-1,Math.round(e+d/2));this.stroke();this.lineCap="butt"}}},SL:{stone:{draw:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius;this.fillStyle=a.c||p(b,a.x,a.y);this.beginPath();this.rect(c-d/2,e-d/2,d,d);this.fill()}}},SM:{stone:{draw:function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius;this.strokeStyle=a.c||p(b,a.x,a.y);this.lineWidth=2*(a.lineWidth||k("markupLinesWidth",b)||1);this.beginPath();this.arc(c-d/3,e-d/3,d/6,0,2*\nMath.PI,!0);this.stroke();this.beginPath();this.arc(c+d/3,e-d/3,d/6,0,2*Math.PI,!0);this.stroke();this.beginPath();this.moveTo(c-d/1.5,e);this.bezierCurveTo(c-d/1.5,e+d/2,c+d/1.5,e+d/2,c+d/1.5,e);this.stroke()}}},outline:{stone:{draw:function(a,b){this.globalAlpha=a.alpha?a.alpha:.3;a.stoneStyle?h.drawHandlers[a.stoneStyle].stone.draw.call(this,a,b):b.stoneHandler.stone.draw.call(this,a,b);this.globalAlpha=1}}},mini:{stone:{draw:function(a,b){b.stoneRadius/=2;a.stoneStyle?h.drawHandlers[a.stoneStyle].stone.draw.call(this,\na,b):b.stoneHandler.stone.draw.call(this,a,b);b.stoneRadius*=2}}}};h.coordinates={grid:{draw:function(a,b){var c,e,d,f,g,h;this.fillStyle=k("coordinatesColor",b);this.textBaseline="middle";this.textAlign="center";this.font=b.stoneRadius+"px "+(b.font||"");d=b.getX(-.75);f=b.getX(b.size-.25);g=b.getY(-.75);h=b.getY(b.size-.25);for(var l=0;l<b.size;l++)c=l+65,73<=c&&c++,e=b.getY(l),this.fillText(b.size-l,d,e),this.fillText(b.size-l,f,e),e=b.getX(l),this.fillText(String.fromCharCode(c),e,g),this.fillText(String.fromCharCode(c),\ne,h);this.fillStyle="black"}}};h.CanvasLayer=function(){this.element=document.createElement("canvas");this.context=this.element.getContext("2d");this.pixelRatio=v.devicePixelRatio||1;1<this.pixelRatio&&this.context.scale(this.pixelRatio,this.pixelRatio)};h.CanvasLayer.prototype={constructor:h.CanvasLayer,setDimensions:function(a,b){this.element.width=a;this.element.style.width=a/this.pixelRatio+"px";this.element.height=b;this.element.style.height=b/this.pixelRatio+"px"},appendTo:function(a,b){this.element.style.position=\n"absolute";this.element.style.zIndex=b;a.appendChild(this.element)},removeFrom:function(a){a.removeChild(this.element)},getContext:function(){return this.context},draw:function(a){},clear:function(){this.context.clearRect(0,0,this.element.width,this.element.height)}};h.GridLayer=g.extendClass(h.CanvasLayer,function(){this.super.call(this)});h.GridLayer.prototype.draw=function(a){var b;this.context.beginPath();this.context.lineWidth=k("gridLinesWidth",a);this.context.strokeStyle=k("gridLinesColor",\na);var c=Math.round(a.left),e=Math.round(a.top),d=Math.round(a.fieldWidth*(a.size-1)),f=Math.round(a.fieldHeight*(a.size-1));this.context.strokeRect(c-a.ls,e-a.ls,d,f);for(var g=1;g<a.size-1;g++)b=Math.round(a.getX(g))-a.ls,this.context.moveTo(b,e),this.context.lineTo(b,e+f),b=Math.round(a.getY(g))-a.ls,this.context.moveTo(c,b),this.context.lineTo(c+d,b);this.context.stroke();this.context.fillStyle=k("starColor",a);if(a.starPoints[a.size])for(var h in a.starPoints[a.size])this.context.beginPath(),\nthis.context.arc(a.getX(a.starPoints[a.size][h].x)-a.ls,a.getY(a.starPoints[a.size][h].y)-a.ls,k("starSize",a),0,2*Math.PI,!0),this.context.fill()};h.MultipleCanvasLayer=g.extendClass(h.CanvasLayer,function(){this.init(4)});h.MultipleCanvasLayer.prototype.init=function(a){var b,c;this.layers=a;this.elements=[];this.contexts=[];this.pixelRatio=v.devicePixelRatio||1;for(var e=0;e<a;e++)b=document.createElement("canvas"),c=b.getContext("2d"),1<this.pixelRatio&&c.scale(this.pixelRatio,this.pixelRatio),\nthis.elements.push(b),this.contexts.push(c)};h.MultipleCanvasLayer.prototype.appendTo=function(a,b){for(var c=0;c<this.layers;c++)this.elements[c].style.position="absolute",this.elements[c].style.zIndex=b,a.appendChild(this.elements[c])};h.MultipleCanvasLayer.prototype.removeFrom=function(a){for(var b=0;b<this.layers;b++)a.removeChild(this.elements[b])};h.MultipleCanvasLayer.prototype.getContext=function(a){return a.x%2?a.y%2?this.contexts[0]:this.contexts[1]:a.y%2?this.contexts[2]:this.contexts[3]};\nh.MultipleCanvasLayer.prototype.clear=function(a,b){for(var c=0;c<this.layers;c++)this.contexts[c].clearRect(0,0,this.elements[c].width,this.elements[c].height)};h.MultipleCanvasLayer.prototype.setDimensions=function(a,b){for(var c=0;c<this.layers;c++)this.elements[c].width=a,this.elements[c].style.width=a/this.pixelRatio+"px",this.elements[c].height=b,this.elements[c].style.height=b/this.pixelRatio+"px"};h.ShadowLayer=g.extendClass(h.MultipleCanvasLayer,function(a,b,c){this.init(2);this.shadowSize=\nb===q?1:b;this.board=a});h.ShadowLayer.prototype.getContext=function(a){return a.x%2&&a.y%2||!(a.x%2||a.y%2)?this.contexts[0]:this.contexts[1]};h.ShadowLayer.prototype.setDimensions=function(a,b){this.super.prototype.setDimensions.call(this,a,b);for(var c=0;c<this.layers;c++)this.contexts[c].setTransform(1,0,0,1,Math.round(this.shadowSize*this.board.stoneRadius/7),Math.round(this.shadowSize*this.board.stoneRadius/7))};var E=function(a,b){var c=b.getX(a.x),e=b.getY(a.y),d=b.stoneRadius;this.clearRect(c-\n2*d-b.ls,e-2*d-b.ls,4*d,4*d)},z=function(){return 3*this.width/(4*(this.bx+1-this.tx)+2)-this.fieldWidth*this.tx},A=function(){return 3*this.height/(4*(this.by+1-this.ty)+2)-this.fieldHeight*this.ty},B=function(a,b){for(var c,e=0;e<this.obj_arr[a][b].length;e++){var d=this.obj_arr[a][b][e];c=d.type?"string"==typeof d.type?h.drawHandlers[d.type]:d.type:this.stoneHandler;for(var f in c)c[f].clear?c[f].clear.call(this[f].getContext(d),d,this):E.call(this[f].getContext(d),d,this)}},x=function(a,b){for(var c,\ne=0;e<this.obj_arr[a][b].length;e++){var d=this.obj_arr[a][b][e];c=d.type?"string"==typeof d.type?h.drawHandlers[d.type]:d.type:this.stoneHandler;for(var f in c)c[f].draw.call(this[f].getContext(d),d,this)}},F=function(a){var b;b=a.layerX*this.pixelRatio;b-=this.left;b/=this.fieldWidth;b=Math.round(b);a=a.layerY*this.pixelRatio;a-=this.top;a/=this.fieldHeight;a=Math.round(a);return{x:b>=this.size?-1:b,y:a>=this.size?-1:a}},C=function(){this.element.style.width=this.width/this.pixelRatio+"px";this.element.style.height=\nthis.height/this.pixelRatio+"px";this.stoneRadius=k("stoneSize",this);this.ls=k("linesShift",this);for(var a=0;a<this.layers.length;a++)this.layers[a].setDimensions(this.width,this.height)};h.prototype={constructor:h,init:function(){this.obj_arr=[];for(var a=0;a<this.size;a++){this.obj_arr[a]=[];for(var b=0;b<this.size;b++)this.obj_arr[a][b]=[]}this.obj_list=[];this.layers=[];this.listeners=[];this.element=document.createElement("div");this.element.className="wgo-board";this.element.style.position=\n"relative";this.background&&("#"==this.background[0]?this.element.style.backgroundColor=this.background:this.element.style.backgroundImage="url(\'"+this.background+"\')");this.grid=new h.GridLayer;this.shadow=new h.ShadowLayer(this,k("shadowSize",this));this.stone=new h.MultipleCanvasLayer;this.addLayer(this.grid,100);this.addLayer(this.shadow,200);this.addLayer(this.stone,300)},setWidth:function(a){this.width=a;this.width*=this.pixelRatio;this.fieldHeight=this.fieldWidth=4*this.width/(4*(this.bx+1-\nthis.tx)+2);this.left=z.call(this);this.height=(this.by-this.ty+1.5)*this.fieldHeight;this.top=A.call(this);C.call(this);this.redraw()},setHeight:function(a){this.height=a;this.height*=this.pixelRatio;this.fieldWidth=this.fieldHeight=4*this.height/(4*(this.by+1-this.ty)+2);this.top=A.call(this);this.width=(this.bx-this.tx+1.5)*this.fieldWidth;this.left=z.call(this);C.call(this);this.redraw()},setDimensions:function(a,b){this.width=a||parseInt(this.element.style.width,10);this.width*=this.pixelRatio;\nthis.height=b||parseInt(this.element.style.height,10);this.height*=this.pixelRatio;this.fieldWidth=4*this.width/(4*(this.bx+1-this.tx)+2);this.fieldHeight=4*this.height/(4*(this.by+1-this.ty)+2);this.left=z.call(this);this.top=A.call(this);C.call(this);this.redraw()},getSection:function(){return this.section},setSection:function(a,b,c,e){this.section="object"==typeof a?a:{top:a,right:b,bottom:c,left:e};this.tx=this.section.left;this.ty=this.section.top;this.bx=this.size-1-this.section.right;this.by=\nthis.size-1-this.section.bottom;this.setDimensions()},setSize:function(a){a=a||19;if(a!=this.size){this.size=a;this.obj_arr=[];for(a=0;a<this.size;a++){this.obj_arr[a]=[];for(var b=0;b<this.size;b++)this.obj_arr[a][b]=[]}this.bx=this.size-1-this.section.right;this.by=this.size-1-this.section.bottom;this.setDimensions()}},redraw:function(){try{for(var a=0;a<this.layers.length;a++)this.layers[a].clear(this),this.layers[a].draw(this);for(a=0;a<this.size;a++)for(var b=0;b<this.size;b++)x.call(this,a,\nb);for(a=0;a<this.obj_list.length;a++){var c=this.obj_list[a],e=c.handler,d;for(d in e)e[d].draw.call(this[d].getContext(c.args),c.args,this)}}catch(f){console.log("WGo board failed to render. Error: "+f.message)}},getX:function(a){return this.left+a*this.fieldWidth},getY:function(a){return this.top+a*this.fieldHeight},addLayer:function(a,b){a.appendTo(this.element,b);a.setDimensions(this.width,this.height);this.layers.push(a)},removeLayer:function(a){var b=this.layers.indexOf(a);0<=b&&(this.layers.splice(b,\n1),a.removeFrom(this.element))},update:function(a){var b;if(a.remove&&"all"==a.remove)this.removeAllObjects();else if(a.remove)for(b=0;b<a.remove.length;b++)this.removeObject(a.remove[b]);if(a.add)for(b=0;b<a.add.length;b++)this.addObject(a.add[b])},addObject:function(a){if(a.constructor==Array)for(var b=0;b<a.length;b++)this.addObject(a[b]);else try{B.call(this,a.x,a.y);for(var b=this.obj_arr[a.x][a.y],c=0;c<b.length;c++)if(b[c].type==a.type){b[c]=a;x.call(this,a.x,a.y);return}a.type?b.push(a):b.unshift(a);\nx.call(this,a.x,a.y)}catch(e){console.log("WGo board failed to render. Error: "+e.message)}},removeObject:function(a){if(a.constructor==Array)for(var b=0;b<a.length;b++)this.removeObject(a[b]);else try{for(var c=0;c<this.obj_arr[a.x][a.y].length;c++)if(this.obj_arr[a.x][a.y][c].type==a.type){b=c;break}b!==q&&(B.call(this,a.x,a.y),this.obj_arr[a.x][a.y].splice(b,1),x.call(this,a.x,a.y))}catch(e){console.log("WGo board failed to render. Error: "+e.message)}},removeObjectsAt:function(a,b){this.obj_arr[a][b].length&&\n(B.call(this,a,b),this.obj_arr[a][b]=[])},removeAllObjects:function(){this.obj_arr=[];for(var a=0;a<this.size;a++){this.obj_arr[a]=[];for(var b=0;b<this.size;b++)this.obj_arr[a][b]=[]}this.redraw()},addCustomObject:function(a,b){this.obj_list.push({handler:a,args:b});this.redraw()},removeCustomObject:function(a,b){for(var c=0;c<this.obj_list.length;c++){var e=this.obj_list[c];if(e.handler==a&&e.args==b)return this.obj_list.splice(c,1),this.redraw(),!0}return!1},addEventListener:function(a,b){var c=\nthis,e={type:a,callback:b,handleEvent:function(a){var e=F.call(c,a);b(e.x,e.y,a)}};this.element.addEventListener(a,e,!0);this.listeners.push(e)},removeEventListener:function(a,b){for(var c=0;c<this.listeners.length;c++){var e=this.listeners[c];if(e.type==a&&e.callback==b)return this.element.removeEventListener(e.type,e,!0),this.listeners.splice(c,1),!0}return!1},getState:function(){return{objects:g.clone(this.obj_arr),custom:g.clone(this.obj_list)}},restoreState:function(a){this.obj_arr=a.objects||\nthis.obj_arr;this.obj_list=a.custom||this.obj_list;this.redraw()}};h.default={size:19,width:0,height:0,font:"Calibri",lineWidth:1,autoLineWidth:!1,starPoints:{19:[{x:3,y:3},{x:9,y:3},{x:15,y:3},{x:3,y:9},{x:9,y:9},{x:15,y:9},{x:3,y:15},{x:9,y:15},{x:15,y:15}],13:[{x:3,y:3},{x:9,y:3},{x:3,y:9},{x:9,y:9}],9:[{x:4,y:4}]},stoneHandler:h.drawHandlers.SHELL,starSize:1,shadowSize:1,stoneSize:1,section:{top:0,right:0,bottom:0,left:0},background:g.DIR+"wood1.jpg",theme:{}};g.Board=h;var s=function(a){this.size=\na||19;this.schema=[];for(a=0;a<this.size*this.size;a++)this.schema[a]=0};s.prototype={constructor:g.Position,get:function(a,b){return 0>a||0>b||a>=this.size||b>=this.size?q:this.schema[a*this.size+b]},set:function(a,b,c){this.schema[a*this.size+b]=c;return this},clear:function(){for(var a=0;a<this.size*this.size;a++)this.schema[a]=0;return this},clone:function(){var a=new s(this.size);a.schema=this.schema.slice(0);return a},compare:function(a){for(var b=[],c=[],e=0;e<this.size*this.size;e++)this.schema[e]&&\n!a.schema[e]?c.push({x:Math.floor(e/this.size),y:e%this.size}):this.schema[e]!=a.schema[e]&&b.push({x:Math.floor(e/this.size),y:e%this.size,c:a.schema[e]});return{add:b,remove:c}}};g.Position=s;var m=function(a,b,c,e){this.size=a||19;this.repeating=b===q?"KO":b;this.allow_rewrite=c||!1;this.allow_suicide=e||!1;this.stack=[];this.stack[0]=new s(this.size);this.stack[0].capCount={black:0,white:0};this.turn=g.B;Object.defineProperty(this,"position",{get:function(){return this.stack[this.stack.length-\n1]},set:function(a){this.stack[this.stack.length-1]=a}})},t=function(a,b,c,e,d){0<=c&&c<a.size&&0<=e&&e<a.size&&a.get(c,e)==d&&(a.set(c,e,0),b.push({x:c,y:e}),t(a,b,c,e-1,d),t(a,b,c,e+1,d),t(a,b,c-1,e,d),t(a,b,c+1,e,d))},u=function(a,b,c,e,d){if(0>c||c>=a.size||0>e||e>=a.size)return!0;if(0==a.get(c,e))return!1;if(!0==b.get(c,e)||a.get(c,e)==-d)return!0;b.set(c,e,!0);return u(a,b,c,e-1,d)&&u(a,b,c,e+1,d)&&u(a,b,c-1,e,d)&&u(a,b,c+1,e,d)},l=function(a,b,c,e){var d=[];if(0<=b&&b<a.size&&0<=c&&c<a.size&&\na.get(b,c)==e){var f=new s(a.size);u(a,f,b,c,e)&&t(a,d,b,c,e)}return d};m.prototype={constructor:m,getPosition:function(){return this.stack[this.stack.length-1]},play:function(a,b,c,e){if(!this.isOnBoard(a,b))return 1;if(!this.allow_rewrite&&0!=this.position.get(a,b))return 2;c||(c=this.turn);var d=this.position.clone();d.set(a,b,c);var f=c,h=l(d,a-1,b,-c).concat(l(d,a+1,b,-c),l(d,a,b-1,-c),l(d,a,b+1,-c));if(!h.length){var k=new s(this.size);if(u(d,k,a,b,c))if(this.allow_suicide)f=-c,t(d,h,a,b,c);\nelse return 3}if(k=this.repeating){a:{var m;if("KO"==this.repeating&&0<=this.stack.length-2)m=this.stack.length-2;else if("ALL"==this.repeating)m=0;else{a=!0;break a}for(var p=this.stack.length-2;p>=m;p--)if(this.stack[p].get(a,b)==d.get(a,b)){for(var k=!0,q=0;q<this.size*this.size;q++)if(this.stack[p].schema[q]!=d.schema[q]){k=!1;break}if(k){a=!1;break a}}a=!0}k=!a}if(k)return 4;if(e)return!1;d.color=c;d.capCount={black:this.position.capCount.black,white:this.position.capCount.white};f==g.B?d.capCount.black+=\nh.length:d.capCount.white+=h.length;this.pushPosition(d);this.turn=-c;return h},pass:function(a){this.pushPosition();a?(this.position.color=a,this.turn=-a):(this.position.color=this.turn,this.turn=-this.turn)},isValid:function(a,b,c){return"number"!=typeof this.play(a,b,c,!0)},isOnBoard:function(a,b){return 0<=a&&0<=b&&a<this.size&&b<this.size},addStone:function(a,b,c){return this.isOnBoard(a,b)&&0==this.position.get(a,b)?(this.position.set(a,b,c||0),!0):!1},removeStone:function(a,b){return this.isOnBoard(a,\nb)&&0!=this.position.get(a,b)?(this.position.set(a,b,0),!0):!1},setStone:function(a,b,c){return this.isOnBoard(a,b)?(this.position.set(a,b,c||0),!0):!1},getStone:function(a,b){return this.isOnBoard(a,b)?this.position.get(a,b):0},pushPosition:function(a){a||(a=this.position.clone(),a.capCount={black:this.position.capCount.black,white:this.position.capCount.white},a.color=this.position.color);this.stack.push(a);a.color&&(this.turn=-a.color);return this},popPosition:function(){var a=null;0<this.stack.length&&\n(a=this.stack.pop(),this.turn=0==this.stack.length?g.B:this.position.color?-this.position.color:-this.turn);return a},firstPosition:function(){this.stack=[];this.stack[0]=new s(this.size);this.stack[0].capCount={black:0,white:0};this.turn=g.B;return this},getCaptureCount:function(a){return a==g.B?this.position.capCount.black:this.position.capCount.white},validatePosition:function(){for(var a,b,c=0,e=0,d=[],f=this.position.clone(),h=0;h<this.size;h++)for(var k=0;k<this.size;k++)if(a=this.position.get(h,\nk))b=d.length,d=d.concat(l(f,h-1,k,-a),l(f,h+1,k,-a),l(f,h,k-1,-a),l(f,h,k+1,-a)),a==g.B?e+=d-b:c+=d-b;this.position.capCount.black+=e;this.position.capCount.white+=c;this.position.schema=f.schema;return d}};g.Game=m;v.WGo=g})(window);\n\n  </script>\n  <style>\n    html, body { margin: 0; padding: 0; background: #07184a; }\n    body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; color: white; }\n    .app { width: 660px; min-height: 830px; background: #07184a; overflow: hidden; }\n    .top-track { height: 82px; display: flex; justify-content: center; align-items: flex-start; gap: 19px; padding-top: 12px; box-sizing: border-box; }\n    .badge-wrap { width: 30px; text-align: center; position: relative; }\n    .badge-num { font-size: 11px; font-weight: 800; margin-bottom: 2px; }\n    .badge { width: 24px; height: 24px; margin: 0 auto; background: #ef314a; clip-path: polygon(50% 0%,61% 33%,96% 35%,68% 56%,79% 91%,50% 70%,21% 91%,32% 56%,4% 35%,39% 33%); position: relative; }\n    .badge::after { content: ""; position: absolute; width: 7px; height: 7px; border: 1px solid white; border-radius: 50%; left: 8px; top: 8px; background: #07184a; }\n    .active-ring { position: absolute; width: 44px; height: 44px; border: 1.5px solid #05b879; border-radius: 50%; left: -7px; top: 12px; }\n    .check { color: #0ac77b; font-weight: 900; font-size: 21px; line-height: 17px; }\n    .board-wrap { display: flex; justify-content: center; padding-top: 96px; }\n    #board { width: 492px; height: 492px; background: #f3c63a; box-shadow: 0 0 0 1px rgba(0,0,0,.15); }\n    #board canvas { cursor: crosshair; }\n    .status { text-align: center; height: 28px; font-size: 16px; font-weight: 800; color: #ffffff; margin-top: 12px; }\n    .controls { display: flex; justify-content: center; align-items: center; gap: 12px; margin-top: 8px; font-size: 14px; }\n    button { border: 0; color: white; font-weight: 800; padding: 8px 14px; border-radius: 4px; cursor: pointer; background: #1c3c88; }\n    button.active { background: #0b9a6b; }\n    button.danger { background: #ef314a; }\n    .captures { min-width: 178px; font-weight: 800; color: #ffffff; }\n    .error { color: #ffda42; }\n    .bottom-icons { height: 58px; margin-top: 48px; color: #ff4a5d; display: flex; align-items: center; justify-content: space-around; font-size: 28px; position: relative; }\n    .home-line { position: absolute; bottom: 8px; left: 226px; width: 208px; height: 4px; border-radius: 4px; background: white; }\n  </style>\n</head>\n<body>\n<div class="app">\n  <div class="top-track">\n    <div class="badge-wrap"><div class="badge-num" style="color:#0ac77b">1</div><div class="active-ring"></div><div class="badge"></div><div class="check">✓</div></div>\n    <div class="badge-wrap"><div class="badge-num">2</div><div class="badge"></div><div class="check">✓</div></div>\n    <div class="badge-wrap"><div class="badge-num">3</div><div class="badge"></div><div class="check">✓</div></div>\n    <div class="badge-wrap"><div class="badge-num">4</div><div class="badge"></div><div class="check">✓</div></div>\n    <div class="badge-wrap"><div class="badge-num">5</div><div class="badge"></div><div class="check">✓</div></div>\n    <div class="badge-wrap"><div class="badge-num">6</div><div class="badge"></div><div class="check">✓</div></div>\n    <div class="badge-wrap"><div class="badge-num">7</div><div class="badge"></div></div>\n  </div>\n  <div class="board-wrap"><div id="board"></div></div>\n  <div id="status" class="status">Loading WGo.js...</div>\n  <div class="controls">\n    <button id="human-first" class="active">Human First</button>\n    <button id="bot-first">Bot First</button>\n    <button id="pass" class="danger">Pass</button>\n    <button id="reset">Reset</button>\n    <span id="captures" class="captures"></span>\n  </div>\n  <div class="bottom-icons"><span>‹</span><span>↻</span><span>⌕</span><span>♙</span><span>♡</span><span>▱</span><span>♡</span><span class="home-line"></span></div>\n</div>\n<script>\n(function() {\n  const N = 9, EMPTY = 0, BLACK = 1, WHITE = -1, KOMI = 2.5, SIMULATIONS = 120;\n  const statusEl = document.getElementById(\'status\'), capturesEl = document.getElementById(\'captures\');\n  const humanFirstBtn = document.getElementById(\'human-first\'), botFirstBtn = document.getElementById(\'bot-first\');\n  const passBtn = document.getElementById(\'pass\'), resetBtn = document.getElementById(\'reset\');\n  if (!window.WGo) { statusEl.innerHTML = \'<span class="error">WGo.js did not load. Check internet access for jsDelivr.</span>\'; return; }\n  const board = new WGo.Board(document.getElementById(\'board\'), {size: N, width: 492, height: 492, stoneHandler: WGo.Board.drawHandlers.SHELL, theme: {gridLinesColor: \'#11151c\', starColor: \'#11151c\', coordinatesColor: \'#11151c\'}});\n  function makeGame() { return {board: Array.from({length: N}, () => Array(N).fill(EMPTY)), player: BLACK, previousBoard: null, lastMove: null, captures: {[BLACK]: 0, [WHITE]: 0}, gameOver: false}; }\n  let game = makeGame(), humanColor = BLACK, botThinking = false;\n  function cloneBoard(b) { return b.map(row => row.slice()); }\n  function copyGame(g) { return {board: cloneBoard(g.board), player: g.player, previousBoard: g.previousBoard ? cloneBoard(g.previousBoard) : null, lastMove: g.lastMove ? g.lastMove.slice() : null, captures: {[BLACK]: g.captures[BLACK], [WHITE]: g.captures[WHITE]}, gameOver: g.gameOver}; }\n  function neighbors(r, c) { const out = []; [[1,0],[-1,0],[0,1],[0,-1]].forEach(([dr, dc]) => { const nr = r + dr, nc = c + dc; if (nr >= 0 && nr < N && nc >= 0 && nc < N) out.push([nr, nc]); }); return out; }\n  function groupAndLiberties(g, r, c) { const color = g.board[r][c], group = new Set(), libs = new Set(), q = [[r, c]]; group.add(`${r},${c}`); while (q.length) { const [x, y] = q.shift(); neighbors(x, y).forEach(([nx, ny]) => { if (g.board[nx][ny] === EMPTY) libs.add(`${nx},${ny}`); else if (g.board[nx][ny] === color && !group.has(`${nx},${ny}`)) { group.add(`${nx},${ny}`); q.push([nx, ny]); } }); } return {group: [...group].map(s => s.split(\',\').map(Number)), libs}; }\n  function boardsEqual(a, b) { return a && b && JSON.stringify(a) === JSON.stringify(b); }\n  function isLegal(g, r, c) { if (g.gameOver || r < 0 || r >= N || c < 0 || c >= N || g.board[r][c] !== EMPTY) return false; const test = copyGame(g); test.board[r][c] = test.player; const opponent = -test.player; neighbors(r, c).forEach(([nr, nc]) => { if (test.board[nr][nc] === opponent) { const res = groupAndLiberties(test, nr, nc); if (res.libs.size === 0) res.group.forEach(([x, y]) => { test.board[x][y] = EMPTY; }); } }); const own = groupAndLiberties(test, r, c); return own.libs.size > 0 && !boardsEqual(test.board, g.previousBoard); }\n  function legalMoves(g) { const moves = []; for (let r = 0; r < N; r++) for (let c = 0; c < N; c++) if (isLegal(g, r, c)) moves.push([r, c]); return moves; }\n  function play(g, r, c) { if (!isLegal(g, r, c)) return false; const oldBoard = cloneBoard(g.board); g.board[r][c] = g.player; const opponent = -g.player; neighbors(r, c).forEach(([nr, nc]) => { if (g.board[nr][nc] === opponent) { const res = groupAndLiberties(g, nr, nc); if (res.libs.size === 0) { g.captures[g.player] += res.group.length; res.group.forEach(([x, y]) => { g.board[x][y] = EMPTY; }); } } }); g.previousBoard = oldBoard; g.lastMove = [r, c]; g.player *= -1; return true; }\n  function score(g) { let blackScore = 0, whiteScore = KOMI; for (let r = 0; r < N; r++) for (let c = 0; c < N; c++) { if (g.board[r][c] === BLACK) blackScore++; if (g.board[r][c] === WHITE) whiteScore++; } return blackScore > whiteScore ? BLACK : WHITE; }\n  function selectMove(g) { const legal = legalMoves(g); if (!legal.length) return null; const scores = new Map(), visits = new Map(); legal.forEach(m => { const k = m.join(\',\'); scores.set(k, 0); visits.set(k, 1); }); const rootPlayer = g.player; for (let i = 0; i < SIMULATIONS; i++) { const move = legal[Math.floor(Math.random() * legal.length)]; const sim = copyGame(g); play(sim, move[0], move[1]); for (let j = 0; j < 60; j++) { const moves = legalMoves(sim); if (!moves.length) break; const m = moves[Math.floor(Math.random() * moves.length)]; play(sim, m[0], m[1]); } const k = move.join(\',\'); visits.set(k, visits.get(k) + 1); if (score(sim) === rootPlayer) scores.set(k, scores.get(k) + 1); } return legal.reduce((best, m) => scores.get(m.join(\',\')) / visits.get(m.join(\',\')) > scores.get(best.join(\',\')) / visits.get(best.join(\',\')) ? m : best, legal[0]); }\n  function drawBoard() { board.removeAllObjects(); const objects = []; for (let r = 0; r < N; r++) for (let c = 0; c < N; c++) if (game.board[r][c] !== EMPTY) objects.push({x: c, y: r, c: game.board[r][c] === BLACK ? WGo.B : WGo.W}); if (objects.length) board.addObject(objects); if (game.lastMove) board.addObject({x: game.lastMove[1], y: game.lastMove[0], type: \'CR\', c: \'#0ac77b\'}); }\n  function draw() { drawBoard(); const turn = game.player === BLACK ? \'Black\' : \'White\'; const side = game.player === humanColor ? \'you\' : \'bot\'; statusEl.textContent = game.gameOver ? \'Game over\' : `${turn}\'s turn (${side})`; capturesEl.textContent = `B Captures: ${game.captures[BLACK]}  W Captures: ${game.captures[WHITE]}`; humanFirstBtn.classList.toggle(\'active\', humanColor === BLACK); botFirstBtn.classList.toggle(\'active\', humanColor === WHITE); }\n  function botMove() { if (game.gameOver || game.player === humanColor) { draw(); return; } botThinking = true; draw(); setTimeout(() => { const move = selectMove(game); if (!move) game.gameOver = true; else play(game, move[0], move[1]); botThinking = false; draw(); }, 30); }\n  function reset(startColor) { humanColor = startColor == null ? humanColor : startColor; game = makeGame(); botThinking = false; draw(); if (humanColor === WHITE) botMove(); }\n  board.addEventListener(\'click\', function(x, y) { if (botThinking || game.gameOver || game.player !== humanColor) return; if (play(game, y, x)) { draw(); botMove(); } else statusEl.textContent = \'Illegal move. Pick another point.\'; });\n  humanFirstBtn.addEventListener(\'click\', () => reset(BLACK)); botFirstBtn.addEventListener(\'click\', () => reset(WHITE)); resetBtn.addEventListener(\'click\', () => reset(humanColor)); passBtn.addEventListener(\'click\', () => { game.gameOver = true; draw(); }); reset(BLACK);\n})();\n</script>\n</body>\n</html>\n'

def show_go_ui():
    srcdoc = _html.escape(GO_APP_HTML, quote=True)
    display(HTML(f'<iframe srcdoc="{srcdoc}" width="680" height="850" style="border:0; background:#07184a;"></iframe>'))


In [6]:
# Final setup cell. Run this after the cells above.
show_go_ui()


/Users/jjburrell/Econometrics/Econometrics/.venv/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
